# Churn Prediction Model & Model Card

This notebook trains a leakage-safe churn model using the provided modeling snapshot.

In [ ]:
import json, pickle
from pathlib import Path
import numpy as np
import pandas as pd
DATA = Path('data/raw')
SNAPSHOT = pd.Timestamp('2025-09-30')
rfm = pd.read_csv(DATA/'rfm_modeling_snapshot.csv', parse_dates=['snapshot_date'])
orders = pd.read_csv(DATA/'orders.csv', parse_dates=['order_date'])
rfm.shape, rfm.split.value_counts().to_dict(), rfm.churn_next_60d.mean()

## Leakage Checks

The raw `orders.csv` includes post-snapshot rows, but this notebook uses the provided modeling snapshot as its feature table. Target and split columns are excluded from predictors.

In [ ]:
feature_cols = [c for c in rfm.columns if c not in ['customer_id','snapshot_date','churn_next_60d','split']]
leakage_audit = {
    'post_snapshot_orders_in_raw_file': int((orders.order_date > SNAPSHOT).sum()),
    'target_excluded': 'churn_next_60d' not in feature_cols,
    'split_excluded': 'split' not in feature_cols,
    'feature_count': len(feature_cols),
}
leakage_audit

## Train / Validation / Test Split

In [ ]:
train = rfm[rfm.split == 'train'].copy()
validation = rfm[rfm.split == 'validation'].copy()
test = rfm[rfm.split == 'test'].copy()
pd.DataFrame({
    'split': ['train','validation','test'],
    'rows': [len(train), len(validation), len(test)],
    'churn_rate': [train.churn_next_60d.mean(), validation.churn_next_60d.mean(), test.churn_next_60d.mean()],
})

## Baseline and Final Model Metrics

In [ ]:
metrics = json.load(open('metrics.json'))
metrics

In [ ]:
summary_rows = []
for model_name in ['baseline_model', 'final_model']:
    for split, vals in metrics[model_name]['metrics_by_split'].items():
        summary_rows.append({'model': model_name, 'split': split, **{k: vals[k] for k in ['accuracy','precision','recall','f1','roc_auc','pr_auc','tp','fp','tn','fn']}})
pd.DataFrame(summary_rows)

## Threshold Selection

The final threshold is chosen on validation data using a business score that weights recall and precision while penalizing outreach volume.

In [ ]:
pd.read_csv('outputs/threshold_search.csv').head(10)

## Feature Importance

In [ ]:
pd.read_csv('outputs/feature_importance.csv').head(20)

## Error Analysis Examples

In [ ]:
pd.read_csv('outputs/error_examples.csv')

## Saved Model Scoring Example

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -35, 35)))

def transform_with_bundle(df, bundle):
    parts = []
    for col in bundle['numeric_cols']:
        x = pd.to_numeric(df[col], errors='coerce').fillna(bundle['medians'][col]).to_numpy(float)
        parts.append(((x - bundle['means'][col]) / bundle['stds'][col]).reshape(-1, 1))
    for col in bundle['categorical_cols']:
        vals = df[col].fillna('Missing').astype(str)
        for cat in bundle['categories'][col]:
            parts.append((vals == cat).astype(float).to_numpy().reshape(-1, 1))
    X = np.hstack(parts)
    return np.hstack([np.ones((len(df), 1)), X])

with open('model.pkl', 'rb') as f:
    model = pickle.load(f)
example = test.head(5).copy()
proba = sigmoid(transform_with_bundle(example, model) @ np.array(model['weights']))
pd.DataFrame({'customer_id': example.customer_id, 'churn_probability': proba, 'predicted_churn': proba >= model['threshold']})

See `error_analysis.md` and `model_card.md` for business interpretation, limitations, ethics, and monitoring.